FILE: flores_phage_pipeline.ipynb
==

In [1]:
#============================================
#1 Imports
#============================================

import os
import shutil
import subprocess
from pathlib import Path

In [2]:
#============================================
#2 Config Setup
#============================================

SRA_ID = "SRR37631993"
THREADS = 2

# adapter auto-detect by fastp
MIN_LENGTH = 100

BASE = Path.cwd().parent

RAW_DIR = BASE / "data/raw"
TRIM_DIR = BASE / "data/trimmed"
ASM_DIR = BASE / "data/assembly"
PHABOX_DIR = BASE / "data/phabox"
LOG_DIR = BASE / "logs"
RESULT_DIR = BASE / "results"

for d in [RAW_DIR, TRIM_DIR, ASM_DIR, PHABOX_DIR, LOG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Directories created")

Directories created


In [3]:
#============================================
#3 Helper Function
#============================================

from IPython.display import display, Markdown


def run_cmd(cmd, step_name=None):
    print("\n" + "="*70)
    if step_name:
        print(f"STEP: {step_name}")
    print("="*70)
    print(cmd)
    print()

    process = subprocess.Popen(
        cmd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        executable="/bin/bash"
    )

    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

    print("\nDONE")

In [4]:
#============================================
#4 Check Dependencies
#============================================

TOOLS = [
    "fastqc",
    "fastp",
    "megahit",
    "multiqc",
    "seqkit",
    "wget"
]

for tool in TOOLS:
    path = shutil.which(tool)
    print(f"{tool}: {'FOUND' if path else 'NOT FOUND'}")

fastqc: FOUND
fastp: FOUND
megahit: FOUND
multiqc: FOUND
seqkit: FOUND
wget: FOUND


In [5]:
#============================================
#5 Disk Space Check
#============================================

usage = shutil.disk_usage("/")

print(f"Total: {usage.total / 1e9:.2f} GB")
print(f"Used : {usage.used / 1e9:.2f} GB")
print(f"Free : {usage.free / 1e9:.2f} GB")

Total: 33.64 GB
Used : 20.31 GB
Free : 11.59 GB


In [7]:
#============================================
#6 Download FASTQ from ENA
#============================================

# ENA direct download links
# Replace if ENA changes mirror/path

R1_URL = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_1.fastq.gz"
R2_URL = "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_2.fastq.gz"

run_cmd(
    f'''wget -c -P {RAW_DIR} {R1_URL}''',
    "Download R1"
)

run_cmd(
    f'''wget -c -P {RAW_DIR} {R2_URL}''',
    "Download R2"
)


STEP: Download R1
wget -c -P /workspaces/flores_phage/data/raw https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_1.fastq.gz

--2026-05-19 17:01:00--  https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_1.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.


DONE

STEP: Download R2
wget -c -P /workspaces/flores_phage/data/raw https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_2.fastq.gz

--2026-05-19 17:01:02--  https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR376/093/SRR37631993/SRR37631993_2.fastq.gz
Resolving ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.sra.ebi.ac.uk (ftp.sra.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 416 

In [8]:
#============================================
#7 Verify Downloaded FASTQ
#============================================

for f in RAW_DIR.glob("*.fastq.gz"):
    print(f"FOUND: {f.name}")

for f in RAW_DIR.glob("*.fastq.gz"):
    size_gb = f.stat().st_size / 1e9
    print(f"{f.name}: {size_gb:.2f} GB")

FOUND: SRR37631993_1.fastq.gz
FOUND: SRR37631993_2.fastq.gz
SRR37631993_1.fastq.gz: 2.74 GB
SRR37631993_2.fastq.gz: 2.76 GB


In [9]:
#============================================
#8 Trimming + QC + Filtering Using fastp
#============================================

R1 = RAW_DIR / f"{SRA_ID}_1.fastq.gz"
R2 = RAW_DIR / f"{SRA_ID}_2.fastq.gz"

TRIM_R1 = TRIM_DIR / f"{SRA_ID}_R1.trimmed.fastq.gz"
TRIM_R2 = TRIM_DIR / f"{SRA_ID}_R2.trimmed.fastq.gz"

run_cmd(
    f'''fastp \
    --in1 {R1} \
    --in2 {R2} \
    --out1 {TRIM_R1} \
    --out2 {TRIM_R2} \
    --thread {THREADS} \
    --detect_adapter_for_pe \
    --length_required {MIN_LENGTH} \
    --compression 6 \
    --html {LOG_DIR}/fastp.html \
    --json {LOG_DIR}/fastp.json''',
    "fastp trimming"
)


STEP: fastp trimming
fastp     --in1 /workspaces/flores_phage/data/raw/SRR37631993_1.fastq.gz     --in2 /workspaces/flores_phage/data/raw/SRR37631993_2.fastq.gz     --out1 /workspaces/flores_phage/data/trimmed/SRR37631993_R1.trimmed.fastq.gz     --out2 /workspaces/flores_phage/data/trimmed/SRR37631993_R2.trimmed.fastq.gz     --thread 2     --detect_adapter_for_pe     --length_required 100     --compression 6     --html /workspaces/flores_phage/logs/fastp.html     --json /workspaces/flores_phage/logs/fastp.json

Detecting adapter sequence for read1...
No adapter detected for read1

Detecting adapter sequence for read2...
No adapter detected for read2

Read1 before filtering:
total reads: 20142515
total bases: 3021377250
Q20 bases: 2978187240(98.5705%)
Q30 bases: 2878736553(95.279%)
Q40 bases: 112695560(3.72994%)

Read2 before filtering:
total reads: 20142515
total bases: 3021377250
Q20 bases: 2974018261(98.4325%)
Q30 bases: 2859019595(94.6264%)
Q40 bases: 105780360(3.50106%)

Read1 aft

In [10]:
#============================================
#9 Remove Raw FASTQ
#============================================

#Optional only to save more disk space

for f in RAW_DIR.glob("*.fastq.gz"):
    os.remove(f)

print("Raw FASTQ removed to save space")

Raw FASTQ removed to save space


In [11]:
#============================================
#10 Optional FastQC
#============================================

run_cmd(
    f'''fastqc \
    {TRIM_DIR}/*.fastq.gz \
    --threads {THREADS} \
    --outdir {LOG_DIR}''',
    "Trimmed FASTQC"
)


STEP: Trimmed FASTQC
fastqc     /workspaces/flores_phage/data/trimmed/*.fastq.gz     --threads 2     --outdir /workspaces/flores_phage/logs

application/gzip
Started analysis of SRR37631993_R1.trimmed.fastq.gz
application/gzip
Started analysis of SRR37631993_R2.trimmed.fastq.gz
Approx 5% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 5% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 10% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 10% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 15% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 15% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 20% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 20% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 25% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 25% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 30% complete for SRR37631993_R2.trimmed.fastq.gz
Approx 30% complete for SRR37631993_R1.trimmed.fastq.gz
Approx 35% complete for SRR37631993_R2.trimmed.fas

In [ ]:
#============================================
#11 Optional MultiQC
#============================================

run_cmd(
    f'''multiqc \
    {LOG_DIR} \
    -o {RESULT_DIR}''',
    "MultiQC"
)

In [17]:
#============================================
#12 Assembly with MEGAHIT
#============================================

run_cmd(
    f'''megahit \
    -1 {TRIM_R1} \
    -2 {TRIM_R2} \
    -o {ASM_DIR} \
    -t {THREADS} \
    --min-contig-len 1000 \
    --out-prefix assembly''',
    "MEGAHIT Assembly"
)


STEP: MEGAHIT Assembly
megahit     -1 /workspaces/flores_phage/data/trimmed/SRR37631993_R1.trimmed.fastq.gz     -2 /workspaces/flores_phage/data/trimmed/SRR37631993_R2.trimmed.fastq.gz     -o /workspaces/flores_phage/data/assembly     -t 2     --min-contig-len 1000     --out-prefix assembly

2026-05-19 17:44:08 - MEGAHIT v1.2.9
2026-05-19 17:44:08 - Using megahit_core with POPCNT and BMI2 support
2026-05-19 17:44:08 - Convert reads to binary library
2026-05-19 17:46:48 - b'INFO  sequence/io/sequence_lib.cpp  :   75 - Lib 0 (/workspaces/flores_phage/data/trimmed/SRR37631993_R1.trimmed.fastq.gz,/workspaces/flores_phage/data/trimmed/SRR37631993_R2.trimmed.fastq.gz): pe, 40279070 reads, 150 max length'
2026-05-19 17:46:48 - b'INFO  utils/utils.h                 :  152 - Real: 160.0519\tuser: 36.1100\tsys: 9.9495\tmaxrss: 256460'
2026-05-19 17:46:48 - k-max reset to: 141 
2026-05-19 17:46:48 - Start assembly. Number of CPU threads 2 
2026-05-19 17:46:48 - k list: 21,29,39,59,79,99,119,141 

RuntimeError: Command failed: megahit     -1 /workspaces/flores_phage/data/trimmed/SRR37631993_R1.trimmed.fastq.gz     -2 /workspaces/flores_phage/data/trimmed/SRR37631993_R2.trimmed.fastq.gz     -o /workspaces/flores_phage/data/assembly     -t 2     --min-contig-len 1000     --out-prefix assembly

In [ ]:
#============================================
#13 Assembly Summary
#============================================

CONTIGS = ASM_DIR / "assembly.contigs.fa"

run_cmd(
    f"seqkit stats {CONTIGS}",
    "Assembly Statistics"
)

In [ ]:
#============================================
#14 PhaBOX
#============================================

print("Assembly file ready for upload:")
print(CONTIGS)

"""
UPLOAD:
assembly.contigs.fa

TO:
https://phage.ee.cityu.edu.hk/phabox

SELECT:
- Virus identification
- Taxonomy
- Host prediction

OPTIONAL:
- CheckV
- iPHoP
- vConTACT2
"""